# Non-invasive sex prediction in *Varanus salvator* from morphometrics
### Rigorous benchmarking, uncertainty-aware prediction and explanation stability

Reproducible notebook that regenerates **every table and figure** for a manuscript built on the public dataset of Alymann et al. (2024, *Scientific Data* 11:337; Figshare doi:10.6084/m9.figshare.24558595, CC BY 4.0).

**What this notebook adds over the original data descriptor**

| # | Contribution | Section |
|---|---|---|
| 1 | Data-integrity audit (derived-feature consistency, implausible variables) | 2 |
| 2 | Sex dimorphism after size adjustment (allometry / ANCOVA) | 3 |
| 3 | Leakage-free features; how noisy a single 70/30 split is (replication of the original design) | 4-5 |
| 4 | Nested, repeated CV benchmark of 17 models with corrected resampled tests | 6 |
| 5 | Feature-set ablation + in-fold minimal field-measurement set | 7-8 |
| 6 | Learning curve, size-shift robustness | 9-10 |
| 7 | Population sex-ratio estimation with misclassification correction | 11 |
| 8 | Conformal prediction with abstention; permutation test | 12-13 |
| 9 | Explanation stability and faithfulness (SHAP, permutation, EBM) | 14 |
| 10 | Error analysis, external-validation module, field tool, packaging | 15-18 |

**How to run.** `FAST = True` (default) is a smoke test (minutes). For the paper set `FAST = False` (my rough estimate: a few hours on 8 cores, longer on a 2-core Colab; I have only timed the smoke test. On small machines use `N_REPEATS = 5` and `MAX_GRID = 6`). Results are cached in `paper_assets/cache`, so a disconnected session resumes where it stopped. Outputs go to `paper_assets/figures` (PNG+PDF, 300 dpi) and `paper_assets/tables` (CSV+LaTeX).

In [ ]:
# Colab / Kaggle: uncomment once
# !pip install -q shap xgboost lightgbm catboost interpret statsmodels

In [ ]:
import os, sys, json, time, warnings, itertools, math, platform, hashlib, glob, zipfile, pickle
from pathlib import Path
from collections import Counter
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from joblib import Parallel, delayed
from IPython.display import display
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, VotingClassifier, StackingClassifier)
from sklearn.model_selection import (StratifiedKFold, StratifiedShuffleSplit, GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, train_test_split, permutation_test_score)
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, matthews_corrcoef, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss, log_loss, roc_curve, precision_recall_curve)
import shap
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from interpret.glassbox import ExplainableBoostingClassifier

# ------------------------------ CONFIG ------------------------------
FAST        = False           # True = smoke test; False = full paper run
SEED        = 42
N_REPEATS   = 2 if FAST else 10      # outer repeats of 5-fold CV (50 outer folds in the full run)
OUTER_K, INNER_K = 5, 3
MAX_GRID    = 3 if FAST else 12      # max hyper-parameter settings tried per inner search
N_JOBS      = -1
N_BOOT      = 200 if FAST else 2000
N_PERM      = 30 if FAST else 500
N_SPLITS_REPLICATION = 40 if FAST else 500
USE_CACHE   = True
USE_TABPFN  = False           # optional: pip install tabpfn (needs weights download)
EXTERNAL_CSV = None           # path to an external validation CSV (see Section 16)
# --------------------------------------------------------------------
CFG_SIG = f"fast{int(FAST)}-seed{SEED}-R{N_REPEATS}-g{MAX_GRID}-sk{sklearn.__version__}"
OUT = Path('paper_assets'); FIG = OUT/'figures'; TAB = OUT/'tables'; CACHE = OUT/'cache'
for p in (FIG, TAB, CACHE): p.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
mpl.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'axes.spines.top': False, 'axes.spines.right': False})
SEXCOL = {'Female': '#CC79A7', 'Male': '#0072B2'}      # Okabe-Ito, colour-blind safe
rng_global = np.random.default_rng(SEED)

def savefig(name, fig=None):
    fig = fig or plt.gcf(); fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(FIG/f'{name}.{ext}', dpi=300, bbox_inches='tight')
    plt.show()

def savetab(df_, name, caption='', index=True, show=True):
    df_.to_csv(TAB/f'{name}.csv', index=index)
    try: df_.to_latex(TAB/f'{name}.tex', index=index, float_format='%.3f', caption=caption, escape=True)
    except Exception as e: print('LaTeX export skipped:', e)
    if show: display(df_)

print('sklearn', sklearn.__version__, '| shap', shap.__version__, '| pandas', pd.__version__, '| FAST =', FAST)

## 1. Data loading
Positive class = **Female (1)**, Male = 0 (as in the dataset). Column names are assigned by position (the file has the 20 columns of Table 3 of the paper in a fixed order) so the notebook is robust to header typos.

In [ ]:
CANDIDATES = [os.environ.get('VARANUS_CSV', ''),
              '/mnt/user-data/uploads/Varanus_salvator_morphology_sex_prediction__19_Features_.csv',
              'Varanus_salvator_morphology_sex_prediction__19_Features_.csv',
              *glob.glob('/kaggle/input/**/*aranus*.csv', recursive=True),
              *glob.glob('/content/*aranus*.csv'), *glob.glob('*aranus*.csv')]
DATA_PATH = next((p for p in CANDIDATES if p and os.path.exists(p)), None)
if DATA_PATH is None:
    try:
        from google.colab import files; DATA_PATH = list(files.upload())[0]
    except Exception:
        raise FileNotFoundError('Put the Figshare CSV next to the notebook or set env VARANUS_CSV.')

SHORT = ['BodyCond','BTC','BTC_STL','BodySize_PCA','EEL','EEL_SL','EEL_STL','SL','SL_STL','SW','SW_STL','SW_SL',
         'STL','SVL','TL','TL_STL','TW','TW_STL','Weight','Sex']
raw_df = pd.read_csv(DATA_PATH)
assert raw_df.shape[1] == 20, f'expected 20 columns, got {raw_df.shape[1]}'
ORIG_COLS = list(raw_df.columns)
df = raw_df.copy(); df.columns = SHORT
df['SexLabel'] = df['Sex'].map({0: 'Male', 1: 'Female'})
FEATURES19 = SHORT[:-1]
X_all = df[FEATURES19].copy(); y = df['Sex'].values.astype(int); n = len(df)
print(DATA_PATH, df.shape, '| Female:', int(y.sum()), 'Male:', int((1-y).sum()))
display(df.head(3))

## 2. Data-integrity audit (Table S1)
Checks that engineered columns equal the formulas in Table 3 of the original paper, and screens for implausible values. Findings go into the *Methods/Limitations* of the manuscript and should be confirmed with the dataset authors.

In [ ]:
def max_rel_err(a, b): return float(np.max(np.abs(a - b) / np.abs(b)))
derived = {'BTC_STL = BTC/STL': max_rel_err(df.BTC_STL, df.BTC/df.STL),
           'EEL_SL = EEL/SL': max_rel_err(df.EEL_SL, df.EEL/df.SL),
           'EEL_STL = EEL/STL': max_rel_err(df.EEL_STL, df.EEL/df.STL),
           'SL_STL = SL/STL': max_rel_err(df.SL_STL, df.SL/df.STL),
           'SW_STL = SW/STL': max_rel_err(df.SW_STL, df.SW/df.STL),
           'SW_SL = SW/SL': max_rel_err(df.SW_SL, df.SW/df.SL),
           'TL_STL = TL/STL': max_rel_err(df.TL_STL, df.TL/df.STL),
           'TW_STL = TW/STL': max_rel_err(df.TW_STL, df.TW/df.STL),
           'BodyCond = Weight/STL': max_rel_err(df.BodyCond, df.Weight/df.STL)}
rho_idx = stats.spearmanr(np.arange(n), y)[0]
RAWV = ['BTC','EEL','SL','SW','STL','SVL','TL','TW','Weight']
mad_out = {c: int((np.abs(df[c]-df[c].median())/(1.4826*stats.median_abs_deviation(df[c]))>4).sum()) for c in RAWV}
audit_rows = [('Animals (n)', n), ('Females / Males', f'{int(y.sum())} / {int((1-y).sum())}'),
              ('Missing values', int(df[FEATURES19].isna().sum().sum())),
              ('Duplicate rows', int(df.duplicated().sum())),
              ('max |STL-(SVL+TL)| (cm)', float(np.abs(df.STL-df.SVL-df.TL).max())),
              ('Rows with SW > SL (%)', round(100*float((df.SW > df.SL).mean()), 1)),
              ('Spearman(row index, sex) [row-order leakage check]', round(float(rho_idx), 3)),
              ('Robust outliers |z_MAD|>4', {k: v for k, v in mad_out.items() if v})]
audit_rows += [(f'Max relative error: {k}', f'{v:.2e}') for k, v in derived.items()]
audit = pd.DataFrame(audit_rows, columns=['Check', 'Result']); audit['Result'] = audit['Result'].astype(str)
savetab(audit, 'TableS1_data_audit', 'Data-integrity audit.', index=False)
if (df.SW > df.SL).mean() > 0.5:
    print('NOTE: "skull width" exceeds "skull length" in most rows -> it is probably a circumference or was measured differently.\n'
          'Report this as a limitation and ask the authors; SW-based ratios should be interpreted with care.')
print('Sorted by sex?', abs(rho_idx) > 0.5, '(if True never use un-shuffled splits)')

## 3. Sexual dimorphism: effect sizes and size-adjusted differences
Table 1 gives raw differences (Cohen's *d*, Female − Male, bootstrap CI; Welch and Mann-Whitney tests, Holm-adjusted). Table 2 asks the biological question that matters for a field method: **is there a sex difference in a trait after accounting for body size?** (log-log OLS of the trait on log SVL with a sex term and sex × size interaction, HC3 robust SEs.)

In [ ]:
def cohens_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp
def boot_d(a, b, B=N_BOOT, seed=SEED):
    r = np.random.default_rng(seed)
    ds = [cohens_d(r.choice(a, len(a)), r.choice(b, len(b))) for _ in range(B)]
    return np.percentile(ds, [2.5, 97.5])

Fm, Ml = df[df.Sex == 1], df[df.Sex == 0]
rows = []
for c in FEATURES19:
    a, b = Fm[c].values, Ml[c].values
    lo, hi = boot_d(a, b)
    rows.append(dict(Feature=c, Female=f'{a.mean():.3f} ± {a.std(ddof=1):.3f}', Male=f'{b.mean():.3f} ± {b.std(ddof=1):.3f}',
                     d=cohens_d(a, b), d_lo=lo, d_hi=hi, p_welch=stats.ttest_ind(a, b, equal_var=False).pvalue,
                     p_mwu=stats.mannwhitneyu(a, b).pvalue))
T1 = pd.DataFrame(rows).set_index('Feature')
T1['p_holm'] = multipletests(T1.p_welch, method='holm')[1]
T1 = T1.sort_values('d', key=np.abs, ascending=False)
savetab(T1.round(4), 'Table1_group_differences', 'Female vs male: mean ± SD, Cohen d (F-M) with bootstrap 95% CI, tests.')

In [ ]:
ALLO = ['BTC','TL','SL','SW','EEL','TW','Weight']
rows = []
for t in ALLO:
    d_ = pd.DataFrame({'y': np.log(df[t]), 'x': np.log(df.SVL), 's': df.Sex})
    m = smf.ols('y ~ x * C(s)', d_).fit(cov_type='HC3')
    rows.append(dict(Trait=t, slope_Male=m.params['x'], slope_diff_F_minus_M=m.params['x:C(s)[T.1]'],
                     p_slope_diff=m.pvalues['x:C(s)[T.1]'], sex_effect_pct=100*(np.exp(m.params['C(s)[T.1]'])-1),
                     p_sex=m.pvalues['C(s)[T.1]'], R2=m.rsquared))
    m2 = smf.ols('y ~ x + C(s)', d_).fit(cov_type='HC3')     # additive model: size-adjusted sex shift
    rows[-1].update(adj_sex_effect_pct=100*(np.exp(m2.params['C(s)[T.1]'])-1), p_adj_sex=m2.pvalues['C(s)[T.1]'])
T2 = pd.DataFrame(rows).set_index('Trait')
T2['p_adj_sex_holm'] = multipletests(T2.p_adj_sex, method='holm')[1]
savetab(T2.round(4), 'Table2_allometry_ancova',
        'Log-log allometry on SVL. adj_sex_effect_pct = % difference in females vs males at equal SVL (additive model).')

In [ ]:
fig = plt.figure(figsize=(11, 8.2)); gs = fig.add_gridspec(2, 2)
# (a) weight distribution / size-class composition
ax = fig.add_subplot(gs[0, 0])
for s, c in SEXCOL.items(): ax.hist(df.loc[df.SexLabel == s, 'Weight'], bins=14, alpha=.65, color=c, label=s)
ax.axvline(5, ls='--', c='k', lw=1); ax.text(5.05, ax.get_ylim()[1]*.9, 'factory limit 5 kg', fontsize=8)
ax.set(xlabel='Weight (kg)', ylabel='Animals', title='(a) Sampled size range'); ax.legend()
# (b) forest of effect sizes
ax = fig.add_subplot(gs[0, 1]); Tp = T1.sort_values('d')
ax.errorbar(Tp.d, range(len(Tp)), xerr=[Tp.d-Tp.d_lo, Tp.d_hi-Tp.d], fmt='o', c='k', ms=4, lw=1)
ax.axvline(0, c='grey', lw=.8); ax.set_yticks(range(len(Tp))); ax.set_yticklabels(Tp.index, fontsize=8)
ax.set(xlabel="Cohen's d (Female - Male), 95% bootstrap CI", title='(b) Effect sizes')
# (c) allometry
ax = fig.add_subplot(gs[1, 0])
for s, c in SEXCOL.items():
    d_ = df[df.SexLabel == s]; ax.scatter(np.log(d_.SVL), np.log(d_.BTC), s=14, color=c, alpha=.7, label=s)
    k, b0 = np.polyfit(np.log(d_.SVL), np.log(d_.BTC), 1); xs = np.linspace(np.log(df.SVL.min()), np.log(df.SVL.max()), 50)
    ax.plot(xs, b0 + k*xs, color=c, lw=2)
ax.set(xlabel='log SVL', ylabel='log BTC', title='(c) Tail-base circumference vs body length'); ax.legend()
# (d) BTC by sex
ax = fig.add_subplot(gs[1, 1])
sns.violinplot(data=df, x='SexLabel', y='BTC', order=['Female', 'Male'], palette=SEXCOL, inner=None, ax=ax, alpha=.6)
sns.stripplot(data=df, x='SexLabel', y='BTC', order=['Female', 'Male'], color='k', size=2.5, alpha=.5, ax=ax)
ax.set(xlabel='', ylabel='BTC (cm)', title='(d) Base tail circumference')
savefig('Fig1_overview')

In [ ]:
# Redundancy: correlation among raw measurements and variance inflation factors (Fig S1 / Table S2)
Zr = (df[RAWV] - df[RAWV].mean()) / df[RAWV].std()
vif = pd.Series([variance_inflation_factor(Zr.values, i) for i in range(len(RAWV))], index=RAWV, name='VIF')
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
sns.heatmap(df[RAWV + ['Sex']].corr(method='spearman'), annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax[0], annot_kws={'size': 7})
ax[0].set_title('(a) Spearman correlation (raw measurements, Sex)')
vif.sort_values().plot.barh(ax=ax[1], color='grey'); ax[1].axvline(10, ls='--', c='r'); ax[1].set_title('(b) VIF (red: 10)')
savefig('FigS1_correlation_vif'); savetab(vif.to_frame().round(2), 'TableS2_VIF')

## 4. Leakage-free feature construction
The original dataset contains a **Body size** column computed by PCA on the *whole* dataset and 8 ratio columns derived from raw tape measurements. Computing PCA (or any data-dependent transform) before splitting leaks information from validation animals. Here everything data-dependent is fitted **inside each training fold** by `FeatureBuilder`:

* `paper19`: the 19 published columns as-is (replication)
* `raw`: 9 raw measurements (BTC, EEL, SL, SW, STL, SVL, TL, TW, Weight)
* `raw_ratios`: raw + 8 ratios + body condition + in-fold PCA body size (19 features)
* `allometric`: log SVL + residuals of log trait on log SVL (size-free shape), regression fitted in-fold

In [ ]:
RAW = ['BTC','EEL','SL','SW','STL','SVL','TL','TW','Weight']
PCA_VARS = ['TW','BTC','SL','SW','EEL','SVL','STL','Weight']
ALLO_TRAITS = ['BTC','TL','SL','SW','EEL','TW','Weight']

class FeatureBuilder(BaseEstimator, TransformerMixin):
    """Builds a feature set from the raw measurements; all data-dependent parts are fitted on training data only."""
    def __init__(self, kind='raw', cols=None):
        self.kind = kind; self.cols = cols
    def fit(self, X, y=None):
        X = pd.DataFrame(X).reset_index(drop=True)
        if self.kind == 'raw_ratios':
            self.mu_ = X[PCA_VARS].mean(); self.sd_ = X[PCA_VARS].std(ddof=0)
            self.pca_ = PCA(n_components=1).fit((X[PCA_VARS]-self.mu_)/self.sd_)
            self.sign_ = 1.0 if self.pca_.components_[0].sum() >= 0 else -1.0      # fixed orientation -> stable explanations
        if self.kind == 'allometric':
            self.reg_ = {t: LinearRegression().fit(np.log(X[['SVL']].values), np.log(X[t].values)) for t in ALLO_TRAITS}
        self.feature_names_ = list(self._build(X).columns)
        return self
    def _build(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        if self.kind == 'paper19': out = X[FEATURES19].copy()
        elif self.kind == 'raw': out = X[RAW].copy()
        elif self.kind == 'raw_ratios':
            out = X[RAW].copy()
            for a, b in [('BTC','STL'),('TL','STL'),('TW','STL'),('SL','STL'),('SW','STL'),('EEL','STL')]: out[f'{a}_STL'] = X[a]/X[b]
            out['SW_SL'] = X.SW/X.SL; out['EEL_SL'] = X.EEL/X.SL; out['BodyCond'] = X.Weight/X.STL
            out['BodySize'] = self.sign_*self.pca_.transform((X[PCA_VARS]-self.mu_)/self.sd_)[:, 0]
        elif self.kind == 'allometric':
            out = pd.DataFrame({'logSVL': np.log(X.SVL)})
            for t in ALLO_TRAITS: out[f'res_{t}'] = np.log(X[t].values) - self.reg_[t].predict(np.log(X[['SVL']].values))
        else: raise ValueError(self.kind)
        return out[list(self.cols)] if self.cols is not None else out
    def transform(self, X): return self._build(X)

class DFScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.mu_ = X.mean(); self.sd_ = X.std(ddof=0).replace(0, 1); return self
    def transform(self, X): return (X - self.mu_) / self.sd_

TAIL = {'BTC','BTC_STL','TL','TL_STL','res_BTC','res_TL'}
HEAD = {'SL','SW','EEL','SL_STL','SW_STL','SW_SL','EEL_SL','EEL_STL','res_SL','res_SW','res_EEL'}
def feat_group(f): return 'Tail' if f in TAIL else ('Head' if f in HEAD else 'Body size / mass')

# sanity check: engineered in-fold features reproduce the published ratios
chk = FeatureBuilder('raw_ratios').fit(X_all).transform(X_all)
print('max |BTC_STL - published|:', float(np.abs(chk.BTC_STL.values - df.BTC_STL.values).max()),
      '| corr(in-sample BodySize, published BodySize_PCA): %.4f' % abs(np.corrcoef(chk.BodySize, df.BodySize_PCA)[0, 1]))

## 5. Replicating the original design: how noisy is a single 70/30 split?
The data descriptor validates six classifiers on one 70/30 split (≈ 44 test animals). With so few test animals the sampling error alone is large. We repeat that design for many random splits with the published 19 features and default hyper-parameters.

In [ ]:
def default_six():
    return {'LR': (LogisticRegression(max_iter=5000), True), 'RF': (RandomForestClassifier(200 if FAST else 500, random_state=SEED, n_jobs=1), False),
            'SVM': (SVC(probability=True, random_state=SEED), True),
            'XGB': (XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, eval_metric='logloss', n_jobs=1, verbosity=0, random_state=SEED), False),
            'AdaBoost': (AdaBoostClassifier(random_state=SEED), False), 'GNB': (GaussianNB(), True)}
def one_split(seed):
    tr, te = train_test_split(np.arange(n), test_size=0.3, random_state=seed)      # unstratified, as in a plain 70/30 split
    out = {}
    for name, (est, sc) in default_six().items():
        steps = ([('sc', DFScaler())] if sc else []) + [('clf', est)]
        pipe = Pipeline(steps).fit(X_all.iloc[tr], y[tr])
        out[name] = accuracy_score(y[te], pipe.predict(X_all.iloc[te]))
    return out
t0 = time.time()
rep = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(one_split)(s) for s in range(N_SPLITS_REPLICATION)))
print(f'{N_SPLITS_REPLICATION} splits in {time.time()-t0:.0f}s')
T3 = pd.DataFrame({'median acc': rep.median(), 'p2.5': rep.quantile(.025), 'p97.5': rep.quantile(.975), 'range (pp)': 100*(rep.max()-rep.min())})
savetab(T3.round(3), 'Table3_single_split_variability', 'Accuracy of six classifiers over repeated random 70/30 splits (published 19 features).')
ntest = int(round(.3*n)); print(f'Binomial SD of accuracy at p=0.75 and n_test={ntest}: {100*np.sqrt(.75*.25/ntest):.1f} percentage points')
fig, ax = plt.subplots(figsize=(7, 4)); sns.boxplot(data=rep, color='.8', ax=ax); sns.stripplot(data=rep, color='k', size=2, alpha=.4, ax=ax)
ax.set(ylabel='Test accuracy', title='Single 70/30 split estimates vary widely'); savefig('Fig2_single_split_variability')

## 6. Nested, repeated cross-validation benchmark (17 models)
* Outer loop: repeated stratified 5-fold CV (`N_REPEATS` repeats). **All models see identical outer folds** (paired comparison).
* Inner loop: 3-fold search maximising AUROC on the outer-training data only.
* Feature construction (PCA, ratios) sits inside the pipeline (no leakage).
* Uncertainty: mean with 95% CI from the **Nadeau-Bengio corrected resampled *t*** (accounts for overlapping training sets); paired tests against the logistic-regression baseline, Holm-adjusted.
* Metrics: AUROC, accuracy, balanced accuracy, MCC, F1 (Female = positive), AUPRC, Brier, log-loss, expected calibration error (ECE on pooled out-of-fold predictions).

In [ ]:
NE = 100 if FAST else 300
def get_zoo():
    z = {
     'Dummy':  (DummyClassifier(strategy='prior'), {}, False),
     'LDA':    (LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'), {}, True),
     'LR-L2':  (LogisticRegression(max_iter=5000), {'C': [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}, True),
     'GaussianNB': (GaussianNB(), {}, True),
     'kNN':    (KNeighborsClassifier(), {'n_neighbors': [5, 9, 15, 25], 'weights': ['uniform', 'distance']}, True),
     'SVM-RBF': (SVC(probability=True, random_state=SEED), {'C': [0.3, 1, 3, 10], 'gamma': ['scale', 0.003, 0.01, 0.03]}, True),
     'RF':     (RandomForestClassifier(NE, random_state=SEED, n_jobs=1), {'max_depth': [3, 5, None], 'min_samples_leaf': [1, 3, 5], 'max_features': ['sqrt', 0.5]}, False),
     'ExtraTrees': (ExtraTreesClassifier(NE, random_state=SEED, n_jobs=1), {'max_depth': [3, 5, None], 'min_samples_leaf': [1, 3, 5], 'max_features': ['sqrt', 0.5]}, False),
     'AdaBoost': (AdaBoostClassifier(random_state=SEED), {'n_estimators': [50, 150], 'learning_rate': [0.05, 0.3, 1.0]}, False),
     'GBM':    (GradientBoostingClassifier(random_state=SEED), {'n_estimators': [50, 150], 'learning_rate': [0.03, 0.1], 'max_depth': [1, 2, 3], 'subsample': [0.7, 1.0]}, False),
     'XGBoost': (XGBClassifier(n_estimators=150, subsample=.8, colsample_bytree=.8, eval_metric='logloss', n_jobs=1, verbosity=0, random_state=SEED),
                 {'max_depth': [2, 3, 4], 'learning_rate': [0.03, 0.1], 'n_estimators': [100, 300], 'min_child_weight': [1, 3]}, False),
     'LightGBM': (LGBMClassifier(n_estimators=150, subsample=.8, subsample_freq=1, colsample_bytree=.8, n_jobs=1, verbose=-1, random_state=SEED),
                  {'num_leaves': [4, 8], 'learning_rate': [0.03, 0.1], 'min_child_samples': [3, 8], 'n_estimators': [100, 300]}, False),
     'CatBoost': (CatBoostClassifier(iterations=60 if FAST else 250, verbose=0, thread_count=1, random_seed=SEED, allow_writing_files=False),
                  {'depth': [3, 5], 'learning_rate': [0.03, 0.1], 'l2_leaf_reg': [3, 10]}, False),
     'EBM':    (ExplainableBoostingClassifier(interactions=0, outer_bags=2 if FAST else 8, inner_bags=0, max_rounds=300 if FAST else 2000,
                                              n_jobs=1, random_state=SEED),
                {'learning_rate': [0.01, 0.03], 'min_samples_leaf': [2, 5]}, False),
     'MLP':    (MLPClassifier(max_iter=3000, random_state=SEED), {'hidden_layer_sizes': [(8,), (16,), (32, 16)], 'alpha': [1e-3, 1e-1, 1.0]}, True),
     'RF-Platt': (None, {}, False),     # filled below
     'Vote':   (None, {}, True),
     'Stack':  (None, {}, True),
    }
    rf_small = RandomForestClassifier(NE, max_depth=4, min_samples_leaf=3, random_state=SEED, n_jobs=1)
    z['RF-Platt'] = (__import__('sklearn.calibration', fromlist=['x']).CalibratedClassifierCV(rf_small, method='sigmoid', cv=3), {}, False)
    base = lambda: [('lr', LogisticRegression(C=1, max_iter=5000)), ('svm', SVC(C=1, probability=True, random_state=SEED)),
                    ('rf', RandomForestClassifier(NE, max_depth=4, min_samples_leaf=3, random_state=SEED, n_jobs=1)),
                    ('xgb', XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=.8, eval_metric='logloss', n_jobs=1, verbosity=0, random_state=SEED))]
    z['Vote'] = (VotingClassifier(base(), voting='soft'), {}, True)
    z['Stack'] = (StackingClassifier(base(), final_estimator=LogisticRegression(max_iter=2000), cv=5, stack_method='predict_proba'), {}, True)
    if USE_TABPFN:
        try:
            from tabpfn import TabPFNClassifier; z['TabPFN'] = (TabPFNClassifier(random_state=SEED), {}, True)
        except Exception as e: print('TabPFN unavailable:', e)
    return z
ZOO = get_zoo()
COMPLEXITY = ['LDA','LR-L2','GaussianNB','kNN','SVM-RBF','EBM','RF','ExtraTrees','RF-Platt','AdaBoost','GBM','XGBoost','LightGBM','CatBoost','MLP','Vote','Stack','TabPFN']
print(list(ZOO))

In [ ]:
SPLITS = [list(StratifiedKFold(OUTER_K, shuffle=True, random_state=SEED + r).split(X_all, y)) for r in range(N_REPEATS)]

def ece_score(yt, p, bins=5):
    q = np.quantile(p, np.linspace(0, 1, bins + 1)); idx = np.clip(np.searchsorted(q[1:-1], p, side='right'), 0, bins - 1)
    return float(sum((idx == b).mean() * abs(yt[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any()))

def fold_metrics(yt, p):
    yh = (p >= .5).astype(int); pc = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(Accuracy=accuracy_score(yt, yh), BalAcc=balanced_accuracy_score(yt, yh), MCC=matthews_corrcoef(yt, yh), F1=f1_score(yt, yh),
                AUROC=roc_auc_score(yt, p), AUPRC=average_precision_score(yt, p), Brier=brier_score_loss(yt, pc), LogLoss=log_loss(yt, pc))

def build_pipeline(name, kind, cols=None):
    est, grid, scale = ZOO[name]
    steps = [('fe', FeatureBuilder(kind, cols))] + ([('sc', DFScaler())] if scale else []) + [('clf', clone(est))]
    return Pipeline(steps), {f'clf__{k}': v for k, v in grid.items()}

def make_search(pipe, grid, seed):
    size = int(np.prod([len(v) for v in grid.values()])); cv = StratifiedKFold(INNER_K, shuffle=True, random_state=seed)
    if size <= MAX_GRID: return GridSearchCV(pipe, grid, scoring='roc_auc', cv=cv, n_jobs=1)
    return RandomizedSearchCV(pipe, grid, n_iter=MAX_GRID, scoring='roc_auc', cv=cv, n_jobs=1, random_state=seed)

def _fit_one(name, kind, cols, tr, te, seed, store):
    pipe, grid = build_pipeline(name, kind, cols); params = {}
    if grid:
        s = make_search(pipe, grid, seed).fit(X_all.iloc[tr], y[tr]); best, params = s.best_estimator_, s.best_params_
    else: best = pipe.fit(X_all.iloc[tr], y[tr])
    return dict(p=best.predict_proba(X_all.iloc[te])[:, 1], te=te, tr=tr, params=params, est=best if store else None)

def run_nested(name, kind='raw_ratios', cols=None, store=False):
    key = f"{name}|{kind}|{','.join(cols) if cols else ''}|{CFG_SIG}"
    fn = CACHE / (hashlib.md5(key.encode()).hexdigest()[:14] + '.pkl')
    if USE_CACHE and fn.exists():
        with open(fn, 'rb') as f: r = pickle.load(f)
        if r['ests'] is not None or not store: return r
    tasks = [(r, k, tr, te) for r in range(N_REPEATS) for k, (tr, te) in enumerate(SPLITS[r])]
    out = Parallel(n_jobs=N_JOBS)(delayed(_fit_one)(name, kind, cols, tr, te, SEED + 100*r + k, store) for r, k, tr, te in tasks)
    oof = np.zeros((N_REPEATS, n)); rows = []; ests = []
    for (r, k, tr, te), o in zip(tasks, out):
        oof[r, te] = o['p']; rows.append(dict(repeat=r, fold=k, **fold_metrics(y[te], o['p'])))
        if store: ests.append(dict(est=o['est'], tr=tr, te=te, r=r, k=k))
    folds = pd.DataFrame(rows)
    pooled = pd.DataFrame([dict(repeat=r, ECE=ece_score(y, oof[r]), AUROC_pooled=roc_auc_score(y, oof[r])) for r in range(N_REPEATS)])
    res = dict(oof=oof, folds=folds, pooled=pooled, ests=ests if store else None, params=[o['params'] for o in out], key=key)
    with open(fn, 'wb') as f: pickle.dump(res, f)
    return res

def nb_ci(scores, k=OUTER_K):
    """Mean and 95% CI using the Nadeau-Bengio corrected resampled variance."""
    s = np.asarray(scores, float); J = len(s); v = s.var(ddof=1) if J > 1 else 0.0
    se = np.sqrt((1/J + 1/(k-1)) * v); t = stats.t.ppf(.975, max(J-1, 1)); return s.mean(), s.mean()-t*se, s.mean()+t*se, se
def nb_test(a, b, k=OUTER_K):
    d = np.asarray(a, float) - np.asarray(b, float); J = len(d); v = d.var(ddof=1)
    if v == 0: return d.mean(), 0.0, 1.0
    t = d.mean() / np.sqrt((1/J + 1/(k-1)) * v); return d.mean(), t, 2*stats.t.sf(abs(t), J-1)

In [ ]:
BENCH_KIND = 'raw_ratios'
XAI_MODELS = ['LR-L2', 'RF', 'XGBoost', 'EBM']
RES = {}
t_all = time.time()
for name in ZOO:
    t0 = time.time(); RES[name] = run_nested(name, BENCH_KIND, store=(name in XAI_MODELS))
    print(f"{name:11s} AUROC={RES[name]['folds'].AUROC.mean():.3f}  acc={RES[name]['folds'].Accuracy.mean():.3f}  ({time.time()-t0:.0f}s)")
print(f'benchmark done in {(time.time()-t_all)/60:.1f} min')

In [ ]:
metrics_list = ['AUROC', 'Accuracy', 'BalAcc', 'MCC', 'F1', 'AUPRC', 'Brier', 'LogLoss']
rows = []; num = {}
for name, r in RES.items():
    row = {'Model': name}; nrow = {}
    for m in metrics_list:
        mu, lo, hi, se = nb_ci(r['folds'][m]); row[m] = f'{mu:.3f} [{lo:.3f}, {hi:.3f}]'; nrow[m] = mu; nrow[m + '_se'] = se
    row['ECE'] = f"{r['pooled'].ECE.mean():.3f} ± {r['pooled'].ECE.std(ddof=1) if len(r['pooled'])>1 else 0:.3f}"; nrow['ECE'] = r['pooled'].ECE.mean()
    rows.append(row); num[name] = nrow
BT = pd.DataFrame(rows).set_index('Model'); NUM = pd.DataFrame(num).T
# paired corrected tests vs LR baseline and vs best model (AUROC), Holm adjusted
ref = 'LR-L2'; best_name = NUM.AUROC.drop('Dummy').idxmax()
pv_ref, pv_best, d_ref = {}, {}, {}
for name in RES:
    if name != ref: d_ref[name], _, pv_ref[name] = nb_test(RES[name]['folds'].AUROC, RES[ref]['folds'].AUROC)
    if name != best_name: _, _, pv_best[name] = nb_test(RES[name]['folds'].AUROC, RES[best_name]['folds'].AUROC)
def holm(d_): 
    k = list(d_); adj = multipletests([d_[i] for i in k], method='holm')[1]; return dict(zip(k, adj))
BT['dAUROC vs LR-L2'] = pd.Series(d_ref).round(3); BT['p_Holm vs LR-L2'] = pd.Series(holm(pv_ref)).round(4)
BT[f'p_Holm vs best ({best_name})'] = pd.Series(holm(pv_best)).round(4)
BT = BT.loc[NUM.sort_values('AUROC', ascending=False).index]
savetab(BT, 'Table4_benchmark', 'Nested repeated 5-fold CV. Mean [95% CI, Nadeau-Bengio]. Female = positive class.')

# headline model by the one-standard-error rule: simplest model within 1 SE of the best AUROC
se_best = NUM.loc[best_name, 'AUROC_se']
HEADLINE = next(m for m in COMPLEXITY if m in NUM.index and NUM.loc[m, 'AUROC'] >= NUM.loc[best_name, 'AUROC'] - se_best)
print('best AUROC model:', best_name, '| headline (1-SE rule):', HEADLINE)

In [ ]:
order = [m for m in NUM.sort_values('AUROC').index]
fig, ax = plt.subplots(1, 3, figsize=(14, 5.6), sharey=True)
for a, m in zip(ax, ['AUROC', 'BalAcc', 'MCC']):
    for i, name in enumerate(order):
        mu, lo, hi, _ = nb_ci(RES[name]['folds'][m]); a.errorbar(mu, i, xerr=[[mu-lo], [hi-mu]], fmt='o', c='C3' if name == HEADLINE else 'k', ms=5, lw=1.2)
    a.axvline(NUM.loc['Dummy', m], c='grey', ls='--', lw=.8); a.set_xlabel(m + ' (95% CI)'); a.set_yticks(range(len(order))); a.set_yticklabels(order)
fig.suptitle('Nested repeated CV benchmark (red = headline model; dashed = majority-class baseline)', y=1.0); savefig('Fig3_benchmark')

In [ ]:
top = list(NUM.drop('Dummy').sort_values('AUROC', ascending=False).index[:4]); 
for m in ['LR-L2', HEADLINE]:
    if m not in top: top.append(m)
grid_fpr = np.linspace(0, 1, 101); grid_rec = np.linspace(0, 1, 101)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.4))
for j, name in enumerate(top):
    tprs, precs, rel = [], [], []
    for r in range(N_REPEATS):
        p = RES[name]['oof'][r]; fpr, tpr, _ = roc_curve(y, p); tprs.append(np.interp(grid_fpr, fpr, tpr))
        pr, rc, _ = precision_recall_curve(y, p); precs.append(np.interp(grid_rec, rc[::-1], pr[::-1]))
    c = f'C{j}'; ax[0].plot(grid_fpr, np.mean(tprs, 0), c=c, label=f"{name} ({NUM.loc[name,'AUROC']:.2f})")
    ax[0].fill_between(grid_fpr, np.min(tprs, 0), np.max(tprs, 0), color=c, alpha=.08)
    ax[1].plot(grid_rec, np.mean(precs, 0), c=c)
    # reliability: pooled OOF, 5 equal-mass bins, averaged over repeats
    bx, by = [], []
    for r in range(N_REPEATS):
        p = RES[name]['oof'][r]; q = np.quantile(p, np.linspace(0, 1, 6)); idx = np.clip(np.searchsorted(q[1:-1], p, side='right'), 0, 4)
        bx.append([p[idx == b].mean() for b in range(5)]); by.append([y[idx == b].mean() for b in range(5)])
    ax[2].plot(np.mean(bx, 0), np.mean(by, 0), 'o-', c=c, ms=4)
ax[0].plot([0, 1], [0, 1], 'k--', lw=.7); ax[0].set(xlabel='1 - specificity', ylabel='Sensitivity (female)', title='(a) ROC (mean over repeats; band = range)'); ax[0].legend(fontsize=8, loc='lower right')
ax[1].axhline(y.mean(), c='k', ls='--', lw=.7); ax[1].set(xlabel='Recall', ylabel='Precision', title='(b) Precision-recall')
ax[2].plot([0, 1], [0, 1], 'k--', lw=.7); ax[2].set(xlabel='Mean predicted P(female)', ylabel='Observed fraction female', title='(c) Reliability (5 equal-mass bins)')
savefig('Fig4_roc_pr_calibration')

In [ ]:
# Appendix: most frequently selected hyper-parameters (Table S3)
rows = []
for name, r in RES.items():
    cnt = Counter(json.dumps({k.replace('clf__', ''): (list(v) if isinstance(v, tuple) else v) for k, v in p.items()}, sort_keys=True, default=str) for p in r['params'] if p)
    if cnt: rows.append((name, cnt.most_common(1)[0][0], f'{cnt.most_common(1)[0][1]}/{sum(cnt.values())}'))
savetab(pd.DataFrame(rows, columns=['Model', 'Modal best hyper-parameters', 'Selected in folds']), 'TableS3_hyperparameters', index=False)

## 7. Feature-set ablation
Which information actually carries the sex signal, and does engineering ratios help? Three model families (linear, bagging, boosting) are re-run with nested CV on each feature set. `SVL + Weight only` is the information recorded by most published harvest studies, which matters for external validation (Section 16).

In [ ]:
FEATSETS = {
 'Published 19 columns (global PCA)': ('paper19', None),
 'Raw measurements (9)':             ('raw', None),
 'Raw + ratios + condition + size (in-fold)': ('raw_ratios', None),
 'Size-adjusted residuals + log SVL': ('allometric', None),
 'Tail set (BTC, TL, BTC:STL, TL:STL)': ('raw_ratios', ['BTC', 'TL', 'BTC_STL', 'TL_STL']),
 'BTC only':                         ('raw', ['BTC']),
 'SVL + Weight only':                ('raw', ['SVL', 'Weight']),
}
ABL_MODELS = ['LR-L2', 'RF', 'XGBoost']
ABL = {}
for fs, (kind, cols) in FEATSETS.items():
    for m in ABL_MODELS:
        ABL[(fs, m)] = run_nested(m, kind, cols)
    print(f'{fs:45s}', ' | '.join(f"{m} {ABL[(fs, m)]['folds'].AUROC.mean():.3f}" for m in ABL_MODELS))
tab = pd.DataFrame({m: {fs: '%.3f [%.3f, %.3f]' % nb_ci(ABL[(fs, m)]['folds'].AUROC)[:3] for fs in FEATSETS} for m in ABL_MODELS})
savetab(tab, 'Table5_feature_set_ablation', 'AUROC [95% CI] by feature set and model (nested repeated CV).')

fig, ax = plt.subplots(figsize=(8.5, 4.6))
for j, m in enumerate(ABL_MODELS):
    for i, fs in enumerate(FEATSETS):
        mu, lo, hi, _ = nb_ci(ABL[(fs, m)]['folds'].AUROC); ax.errorbar(mu, i + (j-1)*.22, xerr=[[mu-lo], [hi-mu]], fmt='o', c=f'C{j}', ms=4, label=m if i == 0 else None)
ax.set_yticks(range(len(FEATSETS))); ax.set_yticklabels(list(FEATSETS)); ax.invert_yaxis(); ax.axvline(.5, c='grey', ls='--', lw=.8)
ax.set(xlabel='AUROC (95% CI)', title='Feature-set ablation'); ax.legend(loc='lower right'); savefig('Fig5_feature_ablation')

cand = {fs: v for fs, v in {fs: ABL[(fs, 'LR-L2')]['folds'].AUROC.mean() for fs, (k, c) in FEATSETS.items() if c is None and k != 'paper19'}.items()}
BEST_FS = max(cand, key=cand.get); BEST_KIND = FEATSETS[BEST_FS][0]
print('Best leakage-free full feature set for the linear model:', BEST_FS, '->', BEST_KIND)

## 8. Minimal field-measurement set (nested forward selection)
Greedy forward selection (logistic regression, inner 3-fold AUROC) is run **inside every outer training fold**, so the AUROC-vs-*k* curve is an unbiased estimate of "choose *k* measurements, then predict". The final field set is the *k* chosen by the one-standard-error rule, taking the features most frequently selected across folds. Selection stability is reported.

In [ ]:
SFS_KIND = 'raw_ratios'; KMAX = 4 if FAST else 7
def forward_path(Z, yy, kmax, seed):
    cv = StratifiedKFold(INNER_K, shuffle=True, random_state=seed); sel = []
    for _ in range(kmax):
        best = (-1, None)
        for j in range(Z.shape[1]):
            if j in sel: continue
            s = cross_val_score(LogisticRegression(C=1.0, max_iter=2000), Z[:, sel + [j]], yy, cv=cv, scoring='roc_auc').mean()
            if s > best[0]: best = (s, j)
        sel.append(best[1])
    return sel
def sfs_task(r, k, tr, te, seed):
    fe = FeatureBuilder(SFS_KIND).fit(X_all.iloc[tr]); Ztr_df = fe.transform(X_all.iloc[tr]); sc = DFScaler().fit(Ztr_df)
    Ztr = sc.transform(Ztr_df).values; Zte = sc.transform(fe.transform(X_all.iloc[te])).values
    sel = forward_path(Ztr, y[tr], KMAX, seed); out = {}
    for kk in range(1, KMAX + 1):
        p = LogisticRegression(C=1.0, max_iter=2000).fit(Ztr[:, sel[:kk]], y[tr]).predict_proba(Zte[:, sel[:kk]])[:, 1]
        out[kk] = (roc_auc_score(y[te], p), accuracy_score(y[te], (p >= .5).astype(int)))
    return sel, out, fe.feature_names_
t0 = time.time()
sfs_out = Parallel(n_jobs=N_JOBS)(delayed(sfs_task)(r, k, tr, te, SEED + 7*r + k) for r in range(N_REPEATS) for k, (tr, te) in enumerate(SPLITS[r]))
print(f'forward selection done in {time.time()-t0:.0f}s')
POOL = sfs_out[0][2]
sfs_auc = np.array([[o[1][kk][0] for kk in range(1, KMAX + 1)] for o in sfs_out]); sfs_acc = np.array([[o[1][kk][1] for kk in range(1, KMAX + 1)] for o in sfs_out])
ci_k = [nb_ci(sfs_auc[:, i]) for i in range(KMAX)]
best_k = int(np.argmax([c[0] for c in ci_k])); thr = ci_k[best_k][0] - ci_k[best_k][3]
K_FIELD = max(2, next(i + 1 for i, c in enumerate(ci_k) if c[0] >= thr))
freq_step = pd.DataFrame(0.0, index=POOL, columns=[f'step {i+1}' for i in range(KMAX)])
for o in sfs_out:
    for s, j in enumerate(o[0]): freq_step.iloc[j, s] += 1 / len(sfs_out)
in_top = pd.Series({f: np.mean([POOL.index(f) in o[0][:K_FIELD] for o in sfs_out]) for f in POOL}).sort_values(ascending=False)
FIELD_FEATS = list(in_top.index[:K_FIELD])
print('K_FIELD =', K_FIELD, '| field features:', FIELD_FEATS)
T6 = pd.DataFrame({'k': range(1, KMAX + 1), 'AUROC': ['%.3f [%.3f, %.3f]' % c[:3] for c in ci_k], 'Accuracy': sfs_acc.mean(0).round(3)}).set_index('k')
savetab(T6, 'Table6_forward_selection', 'In-fold forward selection (logistic regression).')
savetab(in_top.to_frame(f'P(selected within first {K_FIELD})').round(3), 'TableS4_selection_stability')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
mu = np.array([c[0] for c in ci_k]); lo = np.array([c[1] for c in ci_k]); hi = np.array([c[2] for c in ci_k])
ax[0].plot(range(1, KMAX + 1), mu, 'o-'); ax[0].fill_between(range(1, KMAX + 1), lo, hi, alpha=.2); ax[0].axvline(K_FIELD, c='r', ls='--')
ax[0].axhline(NUM.loc[HEADLINE, 'AUROC'], c='grey', ls=':'); ax[0].text(KMAX, NUM.loc[HEADLINE, 'AUROC'], ' all features', va='bottom', ha='right', fontsize=8)
ax[0].set(xlabel='Number of measurements k', ylabel='AUROC (nested, 95% CI)', title='(a) Performance vs number of measurements')
fs_ = freq_step.loc[freq_step.sum(1) > 0].sort_values('step 1', ascending=False)
sns.heatmap(fs_, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=ax[1]); ax[1].set_title('(b) How often each feature is chosen at each step')
savefig('Fig6_forward_selection')

RES_FIELD = {m: run_nested(m, SFS_KIND, FIELD_FEATS, store=(m == 'LR-L2')) for m in ['LR-L2', 'RF', 'XGBoost']}
for m, r in RES_FIELD.items(): print(f'field set {FIELD_FEATS} + {m}: AUROC', '%.3f [%.3f, %.3f]' % nb_ci(r['folds'].AUROC)[:3], '| acc %.3f' % r['folds'].Accuracy.mean())
print('Note: the fixed-set estimate above is mildly optimistic (features were chosen on the same data); the unbiased value is the in-fold selection curve at k =', K_FIELD)

## 9. Learning curve: how many animals are needed?
AUROC on a fixed held-out 25 % as a function of training-set size (observed range only). We report the average AUROC gain per doubling of the training set instead of extrapolating: in a full run, power-law asymptote fits hit their upper bound and were not identifiable with n = 146.

In [ ]:
SIZES = [20, 30, 40, 60, 80, 100]; N_LC = 15 if FAST else 100
def lc_task(seed):
    tr, te = next(StratifiedShuffleSplit(1, test_size=.25, random_state=seed).split(X_all, y)); rows = []
    for sz in SIZES:
        sub, _ = train_test_split(tr, train_size=sz, stratify=y[tr], random_state=seed)
        for mname, est in [('Logistic regression', LogisticRegression(C=1, max_iter=3000)), ('Random forest', RandomForestClassifier(100, max_depth=4, min_samples_leaf=3, random_state=seed, n_jobs=1))]:
            pipe = Pipeline([('fe', FeatureBuilder(BEST_KIND)), ('sc', DFScaler()), ('clf', est)]).fit(X_all.iloc[sub], y[sub])
            rows.append(dict(seed=seed, n=sz, model=mname, AUROC=roc_auc_score(y[te], pipe.predict_proba(X_all.iloc[te])[:, 1])))
    return rows
lc = pd.DataFrame([r for rows in Parallel(n_jobs=N_JOBS)(delayed(lc_task)(s) for s in range(N_LC)) for r in rows])
fig, ax = plt.subplots(figsize=(7, 4.5)); rows = []
for j, (m, g) in enumerate(lc.groupby('model')):
    s = g.groupby('n').AUROC.agg(['mean', 'std']); ax.errorbar(s.index, s['mean'], yerr=s['std'], fmt='o-', c=f'C{j}', capsize=2, label=m)
    slopes = np.array([np.polyfit(np.log2(gg.n), gg.AUROC, 1)[0] for _, gg in g.groupby('seed')])      # AUROC gain per doubling of n, per split
    rows.append({'Model': m, **{f'n={k}': f'{v:.3f}' for k, v in s['mean'].items()}, 'gain per doubling [95% interval over splits]': f'{slopes.mean():.3f} [{np.percentile(slopes, 2.5):.3f}, {np.percentile(slopes, 97.5):.3f}]'})
ax.set_xscale('log'); ax.set(xlabel='Training animals (log scale)', ylabel='AUROC on held-out animals', title='Learning curve (observed range only; no extrapolation)'); ax.legend(); savefig('Fig7_learning_curve')
savetab(pd.DataFrame(rows).set_index('Model'), 'Table7_learning_curve', 'Observed AUROC by training-set size and average gain per doubling of n. Extrapolation beyond n=100 is deliberately not reported (power-law fits were unidentifiable).')

## 10. Robustness to a shift in body size (pseudo-external validation)
The sample contains animals ≤ 5 kg from one habitat, so true external validation is impossible with these data alone (see Section 16). As a stress test we **train on two weight terciles and test on the held-out tercile**; this mimics applying the model to smaller or larger animals than it was trained on. The last column shows in-distribution performance on the same animals (from the nested CV) for comparison.

In [ ]:
terc = pd.qcut(df.Weight, 3, labels=False).values; tlab = ['small', 'medium', 'large']
def shift_models():
    mk = lambda kind, cols, est: Pipeline([('fe', FeatureBuilder(kind, cols)), ('sc', DFScaler()), ('clf', est)])
    return {f'LR, all features ({BEST_KIND})': lambda: mk(BEST_KIND, None, LogisticRegression(C=1, max_iter=3000)),
            f'LR, field set ({len(FIELD_FEATS)} features)': lambda: mk(SFS_KIND, FIELD_FEATS, LogisticRegression(C=1, max_iter=3000)),
            'RF, all features': lambda: mk(BEST_KIND, None, RandomForestClassifier(NE, max_depth=4, min_samples_leaf=3, random_state=SEED, n_jobs=1))}
rows = []; rb = np.random.default_rng(SEED)
for hold in range(3):
    tr, te = np.where(terc != hold)[0], np.where(terc == hold)[0]
    for mname, f in shift_models().items():
        p = f().fit(X_all.iloc[tr], y[tr]).predict_proba(X_all.iloc[te])[:, 1]; a = roc_auc_score(y[te], p)
        bs = [roc_auc_score(y[te][i], p[i]) for i in (rb.choice(len(te), len(te)) for _ in range(N_BOOT)) if len(set(y[te][i])) == 2]
        rows.append(dict(held_out=tlab[hold] + f' (n={len(te)})', model=mname, AUROC=a, lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5),
                         Accuracy=accuracy_score(y[te], (p >= .5).astype(int))))
sh = pd.DataFrame(rows)
indist = {tlab[h] + f' (n={(terc==h).sum()})': np.mean([roc_auc_score(y[terc == h], RES[HEADLINE]['oof'][r][terc == h]) for r in range(N_REPEATS)]) for h in range(3)}
sh['in-distribution AUROC (headline, nested CV)'] = sh.held_out.map(indist)
sh['AUROC [95% CI]'] = sh.apply(lambda r: f'{r.AUROC:.3f} [{r.lo:.3f}, {r.hi:.3f}]', axis=1)
savetab(sh[['held_out', 'model', 'AUROC [95% CI]', 'Accuracy', 'in-distribution AUROC (headline, nested CV)']].round(3), 'Table8_size_shift', 'Leave-one-weight-tercile-out validation.', index=False)
fig, ax = plt.subplots(figsize=(8, 4.2)); sns.barplot(data=sh, x='held_out', y='AUROC', hue='model', ax=ax, errorbar=None)
ax.axhline(.5, c='grey', ls='--'); ax.set(xlabel='Held-out weight class', ylim=(.3, 1)); ax.legend(fontsize=8); savefig('Fig8_size_shift')

## 11. From individual sex calls to population sex ratios
The stated purpose of the dataset is monitoring **population sex ratios**. A classifier with imperfect sensitivity/specificity biases the naive ratio estimate towards 50 %. Using the Rogan-Gladen correction, $\hat\pi=(q+\mathrm{Sp}-1)/(\mathrm{Se}+\mathrm{Sp}-1)$ (q = fraction of animals called female), with Se/Sp estimated on an independent validation half, we simulate field samples of size *N* from resampled out-of-fold predictions and compare bias and RMSE of the naive and corrected estimators.

In [ ]:
pred0 = (RES[HEADLINE]['oof'][0] >= .5).astype(int); N_SIM = 100 if FAST else 1000; rs = np.random.default_rng(SEED)
def sim_cell(pi, N):
    naive, corr = [], []
    for _ in range(N_SIM):
        A, B = train_test_split(np.arange(n), test_size=.5, stratify=y, random_state=int(rs.integers(1e9)))
        se = (pred0[A][y[A] == 1] == 1).mean(); sp = (pred0[A][y[A] == 0] == 0).mean()
        Bf, Bm = B[y[B] == 1], B[y[B] == 0]; nf = int(round(N * pi))
        d = np.concatenate([rs.choice(Bf, nf), rs.choice(Bm, N - nf)]); q = (pred0[d] == 1).mean(); naive.append(q)
        corr.append(np.clip((q + sp - 1) / (se + sp - 1), 0, 1) if se + sp - 1 > .1 else np.nan)
    return np.array(naive), np.array(corr)
rows = []
for N_ in (50, 200):
    for pi in (.2, .35, .5, .65, .8):
        a, b = sim_cell(pi, N_)
        rows.append(dict(N=N_, true_pi=pi, bias_naive=a.mean()-pi, bias_corrected=np.nanmean(b)-pi, rmse_naive=np.sqrt(np.mean((a-pi)**2)), rmse_corrected=np.sqrt(np.nanmean((b-pi)**2))))
SR = pd.DataFrame(rows); savetab(SR.round(3), 'Table9_sex_ratio_estimation', 'Bias and RMSE of the estimated proportion of females.', index=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for N_, g in SR.groupby('N'):
    ax[0].plot(g.true_pi, g.bias_naive, 'o-', label=f'naive N={N_}'); ax[0].plot(g.true_pi, g.bias_corrected, 's--', label=f'corrected N={N_}')
    ax[1].plot(g.true_pi, g.rmse_naive, 'o-', label=f'naive N={N_}'); ax[1].plot(g.true_pi, g.rmse_corrected, 's--', label=f'corrected N={N_}')
ax[0].axhline(0, c='k', lw=.7); ax[0].set(xlabel='True fraction female', ylabel='Bias', title='(a) Bias'); ax[1].set(xlabel='True fraction female', ylabel='RMSE', title='(b) RMSE'); ax[0].legend(fontsize=7)
savefig('Fig9_sex_ratio_estimation')

**How large must the validation set be?** The corrected estimator is unbiased but divides by (Se + Sp − 1), which is small for a modest classifier, so errors in Se/Sp estimation are amplified. The parametric simulation below fixes the true Se/Sp at the out-of-fold estimates and varies the size *V* of the validation sample used to estimate them and the size *N* of the field sample.

In [ ]:
se_hat = (pred0[y == 1] == 1).mean(); sp_hat = (pred0[y == 0] == 0).mean(); print(f'Se = {se_hat:.3f}, Sp = {sp_hat:.3f}, Se+Sp-1 = {se_hat+sp_hat-1:.3f}')
rs2 = np.random.default_rng(SEED); REPS = 2000 if FAST else 20000; rows = []
for V in (50, 100, 200, 500, 1000):
    for N_ in (50, 200, 1000):
        for pi in (.35, .5, .65):
            Vf = V // 2; nf = int(round(N_ * pi))
            Se_v = rs2.binomial(Vf, se_hat, REPS) / Vf; Sp_v = rs2.binomial(Vf, sp_hat, REPS) / Vf
            q = (rs2.binomial(nf, se_hat, REPS) + rs2.binomial(N_ - nf, 1 - sp_hat, REPS)) / N_
            den = Se_v + Sp_v - 1; corr = np.where(den > .1, np.clip((q + Sp_v - 1) / np.where(den > .1, den, 1), 0, 1), np.nan)
            rows.append(dict(V=V, N=N_, true_pi=pi, rmse_naive=np.sqrt(np.mean((q - pi) ** 2)), rmse_corrected=np.sqrt(np.nanmean((corr - pi) ** 2))))
VS = pd.DataFrame(rows).groupby(['V', 'N'])[['rmse_naive', 'rmse_corrected']].mean().reset_index()
savetab(VS.round(3), 'Table9b_sex_ratio_validation_size', 'Mean RMSE (over true ratios 0.35/0.5/0.65) of the naive and corrected ratio estimators vs validation-set size V and field sample N.', index=False)
fig, ax = plt.subplots(figsize=(6.5, 4))
for j, (N_, g) in enumerate(VS.groupby('N')):
    ax.plot(g.V, g.rmse_corrected, 'o-', c=f'C{j}', label=f'corrected, N={N_}'); ax.axhline(g.rmse_naive.mean(), c=f'C{j}', ls=':', lw=1)
ax.set_xscale('log'); ax.set(xlabel='Validation animals V used to estimate Se and Sp', ylabel='RMSE of estimated fraction female', title='Dotted: naive estimator'); ax.legend(fontsize=8); savefig('Fig9b_validation_size')

## 12. Uncertainty-aware prediction: conformal sets and abstention
Field users need to know **when not to trust a call**. We use split-conformal prediction with the nonconformity score *1 − P(true class)*, both marginal (standard) and **class-conditional (Mondrian)**. Inside each outer training fold, 30 % of the animals form a calibration set. Reported: empirical coverage (should be ≥ 1 − α), prediction-set composition, and accuracy on the animals receiving a single-label call. A risk-coverage curve (accuracy when abstaining on the least confident animals) is computed from the nested out-of-fold probabilities.

In [ ]:
ALPHAS = [0.05, 0.1, 0.2, 0.3]
def conf_quantile(s, alpha):
    m = len(s); level = np.ceil((m + 1) * (1 - alpha)) / m
    return np.inf if level > 1 else np.quantile(s, level, method='higher')
def conformal_run(r):
    rows = []
    for k, (tr, te) in enumerate(SPLITS[r]):
        fit_i, cal_i = train_test_split(tr, test_size=.3, stratify=y[tr], random_state=SEED + 31*r + k)
        pipe, _ = build_pipeline(HEADLINE, BEST_KIND); pipe.fit(X_all.iloc[fit_i], y[fit_i])
        pc = pipe.predict_proba(X_all.iloc[cal_i])[:, 1]; pt = pipe.predict_proba(X_all.iloc[te])[:, 1]
        s_cal = np.where(y[cal_i] == 1, 1 - pc, pc)
        for a in ALPHAS:
            for mode in ('marginal', 'class-conditional'):
                if mode == 'marginal': q = {0: conf_quantile(s_cal, a), 1: conf_quantile(s_cal, a)}
                else: q = {c: conf_quantile(s_cal[y[cal_i] == c], a) for c in (0, 1)}
                inc1 = (1 - pt) <= q[1]; inc0 = pt <= q[0]          # class 1 (female) / class 0 (male) included?
                size = inc0.astype(int) + inc1.astype(int); yt = y[te]
                cov = np.where(yt == 1, inc1, inc0)
                single = size == 1; corr = np.where(inc1, 1, 0) == yt
                rows.append(dict(repeat=r, fold=k, alpha=a, mode=mode, n_te=len(te), cov=cov.sum(), cov_F=cov[yt == 1].sum(), nF=(yt == 1).sum(),
                                 cov_M=cov[yt == 0].sum(), nM=(yt == 0).sum(), empty=(size == 0).sum(), single=single.sum(), both=(size == 2).sum(),
                                 single_correct=(single & corr).sum()))
    return rows
t0 = time.time(); CF = pd.DataFrame([r for rows in Parallel(n_jobs=N_JOBS)(delayed(conformal_run)(r) for r in range(N_REPEATS)) for r in rows]); print(f'{time.time()-t0:.0f}s')
agg = CF.groupby(['mode', 'alpha', 'repeat'])[['n_te', 'cov', 'cov_F', 'nF', 'cov_M', 'nM', 'empty', 'single', 'both', 'single_correct']].sum().reset_index()
agg['coverage'] = agg['cov'] / agg.n_te; agg['coverage_F'] = agg.cov_F / agg.nF; agg['coverage_M'] = agg.cov_M / agg.nM
agg['single_%'] = 100 * agg.single / agg.n_te; agg['both_%'] = 100 * agg.both / agg.n_te; agg['empty_%'] = 100 * agg.empty / agg.n_te
agg['acc_on_single'] = agg.single_correct / agg.single.replace(0, np.nan)
T10 = agg.groupby(['mode', 'alpha'])[['coverage', 'coverage_F', 'coverage_M', 'single_%', 'both_%', 'empty_%', 'acc_on_single']].mean()
savetab(T10.round(3), 'Table10_conformal', f'Conformal prediction with the headline model ({HEADLINE}); averages over repeats.')

# risk-coverage from nested OOF probabilities (headline model)
conf_ = np.abs(RES[HEADLINE]['oof'] - .5); cov_grid = np.linspace(.2, 1, 17); rc = []
for r in range(N_REPEATS):
    order_ = np.argsort(-conf_[r]); corr_ = ((RES[HEADLINE]['oof'][r] >= .5).astype(int) == y)[order_]
    rc.append([corr_[:max(int(round(c * n)), 1)].mean() for c in cov_grid])
rc = np.array(rc)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for mode, ls in (('marginal', '--'), ('class-conditional', '-')):
    g = T10.loc[mode]; ax[0].plot(1 - g.index, g.coverage, 'o' + ls, label=mode)
ax[0].plot([.65, 1], [.65, 1], 'k:', lw=.8); ax[0].set(xlabel='Target coverage 1 - α', ylabel='Empirical coverage', title='(a) Validity'); ax[0].legend()
g = T10.loc['class-conditional'][['single_%', 'both_%', 'empty_%']]; g.index = [f'α={a}' for a in g.index]; g.plot.bar(stacked=True, ax=ax[1], color=['#009E73', '#E69F00', '#999999'], rot=0)
ax[1].set(ylabel='% of animals', title='(b) Prediction-set composition (class-conditional)'); ax[1].legend(['single label', 'both labels (uncertain)', 'empty'], fontsize=8)
ax[2].plot(cov_grid, rc.mean(0), 'o-'); ax[2].fill_between(cov_grid, rc.min(0), rc.max(0), alpha=.2); ax[2].axhline(max(y.mean(), 1-y.mean()), c='grey', ls='--')
ax[2].set(xlabel='Fraction of animals called (rest: abstain)', ylabel='Accuracy on called animals', title='(c) Risk-coverage (nested OOF)'); savefig('Fig10_conformal_abstention')

## 13. Permutation test (is the signal real?)
Label-permutation test of the headline pipeline's AUROC with 5-fold CV (no tuning). H0: features carry no information about sex.

In [ ]:
pipe_h, _ = build_pipeline(HEADLINE, BEST_KIND)
score, perm_scores, pval = permutation_test_score(pipe_h, X_all, y, scoring='roc_auc', cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_permutations=N_PERM, random_state=SEED, n_jobs=N_JOBS)
print(f'AUROC = {score:.3f}; permutation null mean = {perm_scores.mean():.3f} (95th pct {np.percentile(perm_scores, 95):.3f}); p = {pval:.4f} (n_perm = {N_PERM})')
fig, ax = plt.subplots(figsize=(6, 3.5)); ax.hist(perm_scores, bins=20, color='.7'); ax.axvline(score, c='r'); ax.set(xlabel='AUROC under permuted labels', title=f'Permutation test, p = {pval:.3f}'); savefig('FigS2_permutation_test')

## 14. Explainability: attributions, stability and faithfulness
Explanations are computed **on held-out test folds** of the nested CV for four model families (logistic regression, random forest, XGBoost, EBM). Per outer fold we obtain (i) mean |SHAP| (EBM: exact term contributions), (ii) held-out permutation importance (AUROC drop), (iii) group-wise permutation (tail / head / body-size groups permuted jointly, which is the right way to handle correlated predictors). Because every model is refitted on each fold, we can ask whether the explanation is **stable** (Spearman ρ and top-3 Jaccard between folds, agreement across model families) and **faithful** (deletion curves).

In [ ]:
def pos_class(sv):
    if isinstance(sv, list): return np.asarray(sv[1])
    sv = np.asarray(sv); return sv[:, :, 1] if sv.ndim == 3 else sv
def attributions(name, clf, Ztr, Zte):
    if name == 'LR-L2': ex = shap.LinearExplainer(clf, Ztr); sv = np.asarray(ex.shap_values(Zte)); base = float(np.ravel(ex.expected_value)[0])
    elif name in ('RF', 'XGBoost'): ex = shap.TreeExplainer(clf); sv = pos_class(ex.shap_values(Zte)); base = float(np.ravel(ex.expected_value)[-1])
    elif name == 'EBM': sv = np.asarray(clf.eval_terms(Zte)); base = float(clf.intercept_[0]) if hasattr(clf, 'intercept_') else 0.0
    return sv, base
def perm_importance(clf, Z, yy, n_rep, rg, groups=None):
    base = roc_auc_score(yy, clf.predict_proba(Z)[:, 1]); cols = list(Z.columns)
    units = [[c] for c in cols] if groups is None else [[c for c in cols if feat_group(c) == g] for g in groups]; out = []
    for u in units:
        d_ = []
        for _ in range(n_rep):
            Zp = Z.copy(); perm = rg.permutation(len(Z)); Zp[u] = Z[u].values[perm]; d_.append(base - roc_auc_score(yy, clf.predict_proba(Zp)[:, 1]))
        out.append(np.mean(d_))
    return np.array(out)
GROUPS = ['Tail', 'Head', 'Body size / mass']; XAI = {}
t0 = time.time()
for name in XAI_MODELS:
    recs = []
    for e in RES[name]['ests']:
        pipe = e['est']; prep, clf = pipe[:-1], pipe[-1]
        Ztr = prep.transform(X_all.iloc[e['tr']]); Zte = prep.transform(X_all.iloc[e['te']]); rg = np.random.default_rng(SEED + e['r']*10 + e['k'])
        sv, base = attributions(name, clf, Ztr, Zte)
        recs.append(dict(r=e['r'], k=e['k'], te=e['te'], tr=e['tr'], Zte=Zte, Ztr_mean=Ztr.mean(), sv=sv, base=base, clf=clf,
                         shap=np.abs(sv).mean(0), perm=perm_importance(clf, Zte, y[e['te']], 3 if FAST else 10, rg),
                         gperm=perm_importance(clf, Zte, y[e['te']], 3 if FAST else 10, rg, GROUPS)))
    XAI[name] = recs; print(f'{name}: {len(recs)} estimators explained ({time.time()-t0:.0f}s)')
FN = list(XAI['LR-L2'][0]['Zte'].columns)

In [ ]:
def stack(name, key):
    M = np.array([r[key] for r in XAI[name]]); return M / np.clip(M.sum(1, keepdims=True), 1e-12, None) if key in ('shap',) else M
imp = {(m, 'shap'): stack(m, 'shap') for m in XAI}; imp.update({(m, 'perm'): stack(m, 'perm') for m in XAI})
# ---- consensus importance (normalised mean |SHAP|), Fig 11a
cons = pd.DataFrame({m: imp[(m, 'shap')].mean(0) for m in XAI}, index=FN); cons['mean'] = cons.mean(1); cons = cons.sort_values('mean', ascending=False)
savetab(cons.round(4), 'Table11_consensus_importance', 'Normalised mean |SHAP| (EBM: term contributions) per model; mean over outer folds.')
top_f = list(cons.index[:10]); sd_ = {m: pd.Series(imp[(m, 'shap')].std(0), index=FN) for m in XAI}
fig, ax = plt.subplots(1, 3, figsize=(15, 4.8)); w = .2
for j, m in enumerate(XAI):
    ax[0].barh(np.arange(len(top_f)) + (j-1.5)*w, cons.loc[top_f, m], w, xerr=sd_[m][top_f], label=m, color=f'C{j}', error_kw=dict(lw=.7))
ax[0].set_yticks(range(len(top_f))); ax[0].set_yticklabels(top_f); ax[0].invert_yaxis(); ax[0].legend(fontsize=7); ax[0].set(xlabel='Normalised mean |SHAP| (± SD over folds)', title='(a) Global importance')
gp = pd.DataFrame({m: pd.Series(np.array([r['gperm'] for r in XAI[m]]).mean(0), index=GROUPS) for m in XAI}); gps = pd.DataFrame({m: pd.Series(np.array([r['gperm'] for r in XAI[m]]).std(0), index=GROUPS) for m in XAI})
gp.plot.bar(yerr=gps, ax=ax[1], rot=0, capsize=2, error_kw=dict(lw=.7)); ax[1].set(ylabel='AUROC drop when group permuted', title='(b) Grouped permutation importance'); ax[1].legend(fontsize=7)
# (c) cross-model agreement (mean importance vectors)
agree = pd.DataFrame({m: {m2: stats.spearmanr(imp[(m, 'shap')].mean(0), imp[(m2, 'shap')].mean(0))[0] for m2 in XAI} for m in XAI})
sns.heatmap(agree, annot=True, fmt='.2f', vmin=-.2, vmax=1, cmap='viridis', ax=ax[2], cbar=False); ax[2].set_title('(c) Cross-model rank agreement (Spearman)')
savefig('Fig11_importance_overview'); savetab(agree.round(3), 'Table12_cross_model_agreement')

In [ ]:
# ---- stability across outer folds (Fig 12 / Table 13)
def topk_jaccard(a, b, k=3): A, B = set(np.argsort(-a)[:k]), set(np.argsort(-b)[:k]); return len(A & B) / len(A | B)
rows = []; dist = {}
for m in XAI:
    for meth in ('shap', 'perm'):
        M = imp[(m, meth)]; J = len(M); rho, jac = [], []
        for i, j in itertools.combinations(range(J), 2):
            if XAI[m][i]['te'] is XAI[m][j]['te']: continue
            r_ = stats.spearmanr(M[i], M[j])[0]; rho.append(0 if np.isnan(r_) else r_); jac.append(topk_jaccard(M[i], M[j]))
        top1 = Counter(np.argmax(M, 1)); f1_, c1_ = top1.most_common(1)[0]
        rows.append({'Model': m, 'Method': 'mean |SHAP|' if meth == 'shap' else 'permutation', 'median rho [IQR]': f'{np.median(rho):.2f} [{np.percentile(rho,25):.2f}, {np.percentile(rho,75):.2f}]',
                     'top-3 Jaccard': round(float(np.mean(jac)), 3), 'most frequent #1 feature': f'{FN[f1_]} ({100*c1_/J:.0f}% of folds)'}); dist[(m, meth)] = rho
T13 = pd.DataFrame(rows); savetab(T13, 'Table13_explanation_stability', 'Between-fold stability of feature importance rankings.', index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
dd = pd.DataFrame([{'Model': m, 'Method': 'SHAP' if k == 'shap' else 'Permutation', 'rho': v} for (m, k), vals in dist.items() for v in vals])
sns.boxplot(data=dd, x='Model', y='rho', hue='Method', ax=ax[0], fliersize=1); ax[0].set(ylabel='Spearman ρ between folds', title='(a) Stability of importance ranking')
rk = pd.DataFrame(np.array([stats.rankdata(-v) for v in imp[('XGBoost', 'shap')]]), columns=FN)[list(cons.index[:10])]
sns.boxplot(data=rk, orient='h', ax=ax[1], color='.8', fliersize=1); ax[1].invert_xaxis(); ax[1].set(xlabel='Rank of mean |SHAP| across folds (1 = most important)', title='(b) Rank intervals (XGBoost)'); savefig('Fig12_explanation_stability')

In [ ]:
# ---- OOF SHAP assembly (repeat 0): beeswarm, dependence, local explanations
def assemble(name, rep=0):
    rec = [r for r in XAI[name] if r['r'] == rep]; te = np.concatenate([r['te'] for r in rec])
    return te, np.vstack([r['sv'] for r in rec]), pd.concat([r['Zte'] for r in rec], ignore_index=True), np.array([r['base'] for r in rec for _ in r['te']])
te_x, sv_x, Z_x, _ = assemble('XGBoost')
plt.figure(figsize=(7, 4.8)); shap.summary_plot(sv_x, Z_x, max_display=10, show=False); plt.title('XGBoost: out-of-fold SHAP (log-odds of female)'); savefig('Fig13_shap_beeswarm')
f_top2 = list(pd.Series(np.abs(sv_x).mean(0), index=FN).sort_values(ascending=False).index[:2])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, f in zip(ax, f_top2): shap.dependence_plot(f, sv_x, Z_x, ax=a, show=False)
savefig('Fig14_shap_dependence')

In [ ]:
# ---- local explanations from the (exact) linear SHAP of the logistic regression, repeat 0
te_l, sv_l, Z_l, b_l = assemble('LR-L2'); order_l = np.argsort(te_l); sv_l, Z_l, te_l, b_l = sv_l[order_l], Z_l.iloc[order_l].reset_index(drop=True), te_l[order_l], b_l[order_l]
p0 = RES['LR-L2']['oof'][0]
picks = {'Confident correct female': np.where(y == 1)[0][np.argmax(p0[y == 1])], 'Confident correct male': np.where(y == 0)[0][np.argmin(p0[y == 0])],
         'Female called male': np.where(y == 1)[0][np.argmin(p0[y == 1])], 'Male called female': np.where(y == 0)[0][np.argmax(p0[y == 0])]}
fig, ax = plt.subplots(2, 2, figsize=(11, 7)); ax = ax.ravel()
for a, (title, i) in zip(ax, picks.items()):
    s = pd.Series(sv_l[i], index=FN); s = s.reindex(s.abs().sort_values(ascending=False).index)[:6][::-1]
    a.barh([f'{f} (z={Z_l.loc[i, f]:+.1f})' for f in s.index], s.values, color=['#CC79A7' if v > 0 else '#0072B2' for v in s.values])
    a.axvline(0, c='k', lw=.7); a.set_title(f'{title}\nanimal #{i}: P(female) = {p0[i]:.2f}, truth = {"F" if y[i] else "M"}', fontsize=9); a.set_xlabel('SHAP contribution (log-odds)')
savefig('Fig15_local_explanations')

In [ ]:
# ---- faithfulness: deletion curves (replace top-k features by training mean; lower AUROC faster = more faithful)
def deletion_curve(rec, order_idx, kmax=8):
    Z = rec['Zte'].copy(); mu = rec['Ztr_mean']; yy = y[rec['te']]; out = []
    for k in range(kmax + 1):
        Zk = Z.copy()
        for j in order_idx[:k]: Zk.iloc[:, j] = mu.iloc[j]
        out.append(roc_auc_score(yy, rec['clf'].predict_proba(Zk)[:, 1]))
    return np.array(out)
rows = []; fig, ax = plt.subplots(1, 2, figsize=(11, 4)); rgd = np.random.default_rng(SEED)
for a, name in zip(ax, ['LR-L2', 'XGBoost']):
    curves = {'SHAP order': [], 'Permutation order': [], 'Random order': []}
    for rec in XAI[name]:
        curves['SHAP order'].append(deletion_curve(rec, np.argsort(-rec['shap']))); curves['Permutation order'].append(deletion_curve(rec, np.argsort(-rec['perm'])))
        curves['Random order'].append(np.mean([deletion_curve(rec, rgd.permutation(len(FN))) for _ in range(5)], 0))
    for lab, v in curves.items():
        v = np.array(v); a.plot(range(v.shape[1]), v.mean(0), 'o-', ms=3, label=lab); rows.append({'Model': name, 'Ranking': lab, 'mean AUROC over k=0..8': v.mean(), 'AUROC after removing top-3': v[:, 3].mean()})
    a.axhline(.5, c='grey', ls='--', lw=.7); a.set(xlabel='Number of top features removed', ylabel='Held-out AUROC', title=name); a.legend(fontsize=8)
savefig('Fig16_deletion_faithfulness'); savetab(pd.DataFrame(rows).round(3), 'Table14_faithfulness', 'Deletion-curve faithfulness (lower = explanation identifies the features the model relies on).', index=False)

In [ ]:
# ---- EBM glass-box shape functions + ALE for XGBoost, fitted on all data for display
Z_full = FeatureBuilder(BENCH_KIND).fit(X_all).transform(X_all)      # same feature space as the SHAP analysis
ebm = ExplainableBoostingClassifier(interactions=0, outer_bags=2 if FAST else 16, inner_bags=0, max_rounds=300 if FAST else 3000, n_jobs=1, random_state=SEED).fit(Z_full, y)
names_e = list(ebm.term_names_); imp_e = np.asarray(ebm.term_importances()); topE = [names_e[i] for i in np.argsort(-imp_e)[:4]]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
for a, f in zip(ax, topE):
    i = names_e.index(f); cuts = np.asarray(ebm.bins_[i][0], float); sc = np.asarray(ebm.term_scores_[i], float)[1:-1]
    xs = np.r_[Z_full[f].min(), cuts]; m_ = min(len(xs), len(sc))
    a.step(xs[:m_], sc[:m_], where='post', c='C0'); a.axhline(0, c='k', lw=.6); a.set(xlabel=f, ylabel='Contribution to log-odds(female)', title=f'EBM: {f}')
savefig('Fig17_ebm_shapes')

def ale_1d(model, Z, feat, bins=10):
    q = np.unique(np.quantile(Z[feat], np.linspace(0, 1, bins + 1))); idx = np.clip(np.searchsorted(q, Z[feat], side='right') - 1, 0, len(q) - 2); eff = []
    for b in range(len(q) - 1):
        m = idx == b
        if m.sum() == 0: eff.append(0); continue
        lo, hi = Z[m].copy(), Z[m].copy(); lo[feat] = q[b]; hi[feat] = q[b + 1]
        pl, ph = np.clip(model.predict_proba(lo)[:, 1], 1e-4, 1-1e-4), np.clip(model.predict_proba(hi)[:, 1], 1e-4, 1-1e-4); eff.append(np.mean(np.log(ph/(1-ph)) - np.log(pl/(1-pl))))
    a = np.cumsum(eff); return (q[:-1] + q[1:]) / 2, a - a.mean()
xgb_full = clone(ZOO['XGBoost'][0]).set_params(max_depth=3, learning_rate=0.05, n_estimators=150).fit(Z_full, y)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, f in zip(ax, f_top2 + [[c for c in cons.index if c not in f_top2][0]]):
    x_, e_ = ale_1d(xgb_full, Z_full, f); a.plot(x_, e_, 'o-'); a.axhline(0, c='k', lw=.6); a.set(xlabel=f, ylabel='ALE (centred log-odds)', title=f'ALE: {f}')
savefig('Fig18_ale')

## 15. Error analysis
Where does the model fail? Per-animal misclassification frequency across repeats, error rate by size class, and a list of chronically misclassified animals (candidates for recording errors or genuinely ambiguous morphs).

In [ ]:
pred = (RES[HEADLINE]['oof'] >= .5).astype(int); wrong = (pred != y[None, :]).mean(0)         # per-animal error frequency
cm = np.array([[((pred[r] == c1) & (y == c0)).sum() for c1 in (0, 1)] for r in range(N_REPEATS) for c0 in (0, 1)]).reshape(N_REPEATS, 2, 2).mean(0)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
sns.heatmap(cm, annot=True, fmt='.1f', cmap='Blues', cbar=False, ax=ax[0], xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female']); ax[0].set(xlabel='Predicted', ylabel='True', title=f'(a) Mean confusion matrix ({HEADLINE})')
rows = []
for h in range(3):
    m = terc == h
    bs = [rb.choice(wrong[m], m.sum()).mean() for _ in range(N_BOOT)]; rows.append(dict(size_class=tlab[h], n=int(m.sum()), error=wrong[m].mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5),
                                                                          err_F=wrong[m & (y == 1)].mean(), err_M=wrong[m & (y == 0)].mean()))
ER = pd.DataFrame(rows); ax[1].bar(ER.size_class, ER.error, yerr=[ER.error-ER.lo, ER.hi-ER.error], color='.7', capsize=3); ax[1].set(ylabel='Error rate', title='(b) Error by weight class')
ax[2].hist(wrong, bins=10, color='.6'); ax[2].set(xlabel='Fraction of repeats in which animal is misclassified', ylabel='Animals', title='(c) Hard-to-classify animals'); savefig('Fig19_error_analysis')
savetab(ER.round(3), 'Table15_error_by_size', index=False)
hard = df.loc[wrong >= .8, ['Sex', 'BTC', 'SVL', 'TL', 'STL', 'Weight']].copy(); hard['error_freq'] = wrong[wrong >= .8]; savetab(hard.round(2), 'TableS5_hard_animals', 'Animals misclassified in >= 80% of repeats (check for recording errors).')

## 16. External validation
**Status of external data (searched Oct 2026).** I found no openly downloadable *V. salvator* morphometric dataset that records the same variables with dissection-confirmed sex. Related studies report sex-specific sizes (SVL/mass) for harvested animals (Shine et al. 1996, S. Sumatra *n* = 166; Shine et al. 1998, N. Sumatra *n* = 399; Riau, Indonesia *n* = 660, SVL + body mass, 2025; Mahaprom et al. 2015, Thailand, live-trapped, eye-to-ear difference) but I could not find their individual-level data openly deposited; the Riau article page states its data download is not yet available. They are worth requesting from the authors. Because those studies mainly record **SVL and mass**, the `SVL + Weight only` row of Table 5 shows what an external test with such data could possibly achieve.

The function below validates the trained model on any external CSV once you obtain one: map your column names, and it evaluates (i) AUROC/accuracy/calibration of a model trained on **all** animals here using only the features available externally, and (ii) the shift in feature distributions. If `EXTERNAL_CSV` is `None` it prints a data-collection template.

In [ ]:
def external_validation(path, colmap, sex_col='Sex', female_value=1, feature_kind='raw', min_features=1):
    """colmap: {'your_column': 'SVL', ...} using names from RAW. Trains on this dataset with the intersecting features only."""
    ext = pd.read_csv(path).rename(columns=colmap); feats = [c for c in RAW if c in ext.columns]
    assert len(feats) >= min_features, f'need at least {min_features} mapped features from {RAW}'
    ye = (ext[sex_col] == female_value).astype(int).values; Xe = ext[feats]
    pipe = Pipeline([('fe', FeatureBuilder('raw', feats)), ('sc', DFScaler()), ('clf', LogisticRegression(C=1, max_iter=3000))]).fit(X_all, y)
    pe = pipe.predict_proba(Xe)[:, 1]; bs = [roc_auc_score(ye[i], pe[i]) for i in (np.random.default_rng(0).choice(len(ye), len(ye)) for _ in range(N_BOOT)) if len(set(ye[i])) == 2]
    res = dict(n=len(ye), features=feats, AUROC=roc_auc_score(ye, pe), CI=np.percentile(bs, [2.5, 97.5]), Accuracy=accuracy_score(ye, (pe >= .5).astype(int)), ECE=ece_score(ye, pe),
               prevalence_ext=ye.mean(), prevalence_train=y.mean())
    print(json.dumps(res, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))); return res, pe
if EXTERNAL_CSV:
    ext_res, _ = external_validation(EXTERNAL_CSV, colmap={})          # EDIT colmap, e.g. {'svl_cm': 'SVL', 'mass_kg': 'Weight'}
else:
    template = pd.DataFrame(columns=['animal_id', 'site', 'date', 'observer', 'Sex'] + RAW + ['repeat_BTC', 'repeat_TL', 'sex_confirmation_method'])
    template.to_csv(OUT/'external_validation_template.csv', index=False); print('No external CSV set. Data-collection template saved to', OUT/'external_validation_template.csv')
    print('Request repeated measurements by a second observer (repeat_*) so measurement error can be quantified.')

## 17. Field tool: minimal model with counterfactual margins
A logistic regression on the selected minimal measurements, plus conformal calibration scores from the nested out-of-fold predictions (cross-conformal style) so that the tool can answer **Female / Male / Uncertain**. The *counterfactual margin* is the change in the most important measurement that would flip the call, to be compared with tape-measure error (collect repeatability data first).

In [ ]:
import joblib
field_pipe = Pipeline([('fe', FeatureBuilder(SFS_KIND, FIELD_FEATS)), ('sc', DFScaler()), ('clf', LogisticRegression(C=1.0, max_iter=3000))]).fit(X_all, y)
oof_f = RES_FIELD['LR-L2']['oof'].mean(0); s_cal = np.where(y == 1, 1 - oof_f, oof_f)
FIELD_Q = {a: {c: float(conf_quantile(s_cal[y == c], a)) for c in (0, 1)} for a in (0.1, 0.2)}
def field_predict(new_df, alpha=0.2):
    p = field_pipe.predict_proba(new_df)[:, 1]; out = []
    for pi in p:
        s = ([1] if (1 - pi) <= FIELD_Q[alpha][1] else []) + ([0] if pi <= FIELD_Q[alpha][0] else [])
        out.append('Female' if s == [1] else 'Male' if s == [0] else 'Uncertain')
    return pd.DataFrame({'P_female': p.round(3), 'call': out})
demo = field_predict(X_all.iloc[:8]); demo['truth'] = np.where(y[:8] == 1, 'Female', 'Male'); display(demo)
coef = pd.DataFrame({'std_coef': field_pipe[-1].coef_[0], 'odds_ratio_per_SD': np.exp(field_pipe[-1].coef_[0]), 'SD': field_pipe[1].sd_.values, 'mean': field_pipe[1].mu_.values}, index=FIELD_FEATS)
savetab(coef.round(3), 'Table16_field_model_coefficients', 'Standardised logistic regression of the field model (positive = female).')
# counterfactual margin on the dominant feature (in its own units)
key = coef.std_coef.abs().idxmax(); b_raw = coef.loc[key, 'std_coef'] / coef.loc[key, 'SD']
logit = field_pipe.decision_function(X_all); delta = -logit / b_raw
print(f'Dominant feature: {key}. Median |change| needed to flip a call: {np.median(np.abs(delta)):.3f} (feature units); share of animals within 1 unit: {(np.abs(delta) < 1).mean():.2f}')
fig, ax = plt.subplots(figsize=(6, 3.5)); ax.hist(np.clip(delta, -np.percentile(np.abs(delta), 95), np.percentile(np.abs(delta), 95)), bins=30, color='.6'); ax.axvline(0, c='k')
ax.set(xlabel=f'Change in {key} that flips the call', ylabel='Animals', title='Counterfactual margins'); savefig('Fig20_counterfactual_margin')
joblib.dump(dict(pipeline=field_pipe, conformal_q=FIELD_Q, features=FIELD_FEATS, raw_columns=FEATURES19), OUT/'field_model.joblib')
json.dump(dict(features=FIELD_FEATS, nested_cv_auroc=nb_ci(RES_FIELD['LR-L2']['folds'].AUROC)[:3], n_train=int(n), trained_on='oil palm plantations, Perak, <=5 kg skin-trade animals',
               warning='Not validated externally; do not apply to animals >5 kg or other habitats.'), open(OUT/'field_model_card.json', 'w'), indent=1, default=float)

## 18. Reproducibility package
Collects versions, headline numbers and a zip of every figure and table.

In [ ]:
headline = dict(headline_model=HEADLINE, best_model=best_name, feature_set=BEST_FS, field_features=FIELD_FEATS, K_FIELD=int(K_FIELD),
                AUROC_headline='%.3f [%.3f, %.3f]' % nb_ci(RES[HEADLINE]['folds'].AUROC)[:3], accuracy_headline='%.3f [%.3f, %.3f]' % nb_ci(RES[HEADLINE]['folds'].Accuracy)[:3],
                AUROC_field_set='%.3f [%.3f, %.3f]' % nb_ci(RES_FIELD['LR-L2']['folds'].AUROC)[:3], majority_baseline_acc=float(max(y.mean(), 1-y.mean())),
                permutation_p=float(pval), n=int(n), repeats=N_REPEATS, FAST=FAST, seed=SEED,
                versions=dict(python=platform.python_version(), sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__, shap=shap.__version__))
json.dump(headline, open(OUT/'results_summary.json', 'w'), indent=1, default=str); print(json.dumps(headline, indent=1, default=str))
with zipfile.ZipFile('paper_assets.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in list(FIG.glob('*')) + list(TAB.glob('*')) + [OUT/'results_summary.json', OUT/'field_model_card.json']: z.write(p, p.relative_to(OUT.parent))
print('Figures:', len(list(FIG.glob('*.png'))), '| tables:', len(list(TAB.glob('*.csv'))), '| zip: paper_assets.zip')
if FAST: print('\nFAST mode was on: numbers are smoke-test values. Set FAST = False for the paper run.')

## 19. Figure and table map (for the manuscript)

| Item | Content | Manuscript section |
|---|---|---|
| Table S1 | Data-integrity audit | Methods / Limitations |
| Table 1, Fig 1 | Group differences and effect sizes | Results: dimorphism |
| Table 2 | Size-adjusted (allometric) sex differences | Results: dimorphism |
| Table 3, Fig 2 | Single 70/30 split variability | Results / Discussion (why repeated nested CV) |
| Table 4, Figs 3-4 | 17-model benchmark, ROC / PR / calibration | Results: benchmark |
| Table 5, Fig 5 | Feature-set ablation | Results: which measurements matter |
| Table 6, Fig 6, Table S4 | Minimal field set, selection stability | Results |
| Table 7, Fig 7 | Learning curve | Results / Discussion (data needs) |
| Table 8, Fig 8 | Size-shift stress test | Results: robustness |
| Table 9, Fig 9 | Population sex-ratio estimation | Results: application |
| Table 10, Fig 10 | Conformal sets, abstention | Results: uncertainty |
| Tables 11-14, Figs 11-18 | Attributions, stability, faithfulness, EBM, ALE | Results: explainability |
| Table 15, Fig 19, Table S5 | Error analysis, hard animals | Results / Limitations |
| Table 16, Fig 20, `field_model.joblib` | Field model and counterfactual margins | Results / Data & code availability |

**Suggested framing (title idea):** *"Uncertainty-aware, explanation-stable prediction of sex in the Asian water monitor from field morphometrics."* Lead with: weak-to-moderate signal, honest uncertainty, minimal measurement set, population-level ratio correction; do **not** lead with raw accuracy.

**Checklist before submission:** cite the data descriptor and the Figshare record (CC BY 4.0); deposit this notebook and `paper_assets` on Zenodo/GitHub with the commit hash; follow a prediction-model reporting guideline such as TRIPOD+AI; state the data limitations (≤ 5 kg skin-trade animals, one habitat, a single sampling campaign, suspicious skull-width variable); confirm the ethics statement wording with the dataset's authors; run with `FAST = False` and regenerate all numbers (the manuscript text must quote the full run, not the smoke test).

## 20. Risk register: what this notebook mitigates and what it cannot

| Risk | Mitigation here | Residual risk (say it in the paper) |
|---|---|---|
| No external validation | Size-shift stress test (Sec. 10); ready-to-use external validation function and data template (Sec. 16); permutation test and learning curve | Still single habitat, single campaign. No open external dataset with matching variables was found; request individual-level data from the Sumatra / Riau / Thailand authors or collect new animals. |
| Optimistic single-split evaluation | Repeated stratified nested CV, corrected resampled tests, Table 3 shows the problem | Corrected t-tests are approximate; report all CIs. |
| Leakage (global PCA, ratios, feature selection) | Everything data-dependent is fitted in-fold; forward selection is nested | Final field set is chosen on all data; its fixed-set AUROC is mildly optimistic, quote the nested k-curve. |
| Weak signal, over-claiming | Majority baseline, permutation test, abstention (conformal), accuracy reported with CI | Accuracy may remain moderate; position the tool as a complement to existing methods. |
| Correlated predictors make explanations unreliable | Grouped permutation, stability across folds and across model families, faithfulness curves, glass-box EBM | Attributions describe the model, not causal biology. |
| Population sex-ratio is the real goal | Misclassification-corrected estimator with simulation (Sec. 11) | Se/Sp need local validation; it assumes similar error rates in the field population. |
| Measurement error unknown | Template requests repeated measurements; counterfactual margins to compare with tape error | No repeatability data exist in this dataset. |
| Data quality | Audit (Sec. 2) flags skull width > skull length in all rows; hard-animal list | Query the authors; consider excluding SW-based features in a sensitivity analysis. |
| Size bias (≤ 5 kg) | Weight-tercile shift test; claims restricted to the sampled size range | Do not extrapolate to adults > 5 kg. |